# PROTAC ternary complex: CRBN - dBET23 - BRD4

The whole PROTAC workflow on a system whose answer is known. PDB **6BN7** is a crystal
structure of the E3 ligase **CRBN** and the first bromodomain of **BRD4**, held together by
the PROTAC **dBET23** (JQ1 - octyl linker - thalidomide).

1. assemble dBET23 from three building blocks and check it is the molecule in the crystal,
   stereocentres included;
2. parametrise it;
3. predict the ternary complex from the two proteins, P2Rank pockets and one fixed dBET23
   conformer, and write MD-ready inputs;
4. score the poses against the crystal with [DockQ](https://github.com/bjornwallner/DockQ)
   (CAPRI: >= 0.23 acceptable, >= 0.49 medium, >= 0.80 high).

**Read section 6 before drawing conclusions from the good score.** The conformer used here
was chosen because it reproduces the crystal; with other conformers of the same molecule
the top-ranked pose is usually wrong.

Runtime: about 20 minutes.

## Setup
Checks that PolyLego, GROMACS and AmberTools are available and pins the thread counts.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join("..", "common")))
from polylego_examples import *

# Pin threads for every step launched from this notebook (see examples/README.md).
os.environ.setdefault("OMP_NUM_THREADS", "8")
for k in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[k] = "1"
os.environ.setdefault("PYTHONHASHSEED", "0")

GMX = check_env()
os.environ["POLYLEGO_GMX"] = GMX
# GPU for the GROMACS runs: "auto" (use a compatible GPU if there is one, else CPU),
# "none" (CPU only), or a GPU id such as "0".
GPU = os.environ.get("POLYLEGO_GPU", "auto")
os.makedirs("output", exist_ok=True)

PolyLego   1.4.0  ($CONDA_PREFIX/lib/python3.10/site-packages/PolyLego)
Python     3.10.13
GROMACS    $CONDA_PREFIX/bin/gmx
AmberTools $CONDA_PREFIX/bin/antechamber
OMP_NUM_THREADS        8
OPENBLAS_NUM_THREADS   1
MKL_NUM_THREADS        1


## 1. Building blocks
JQ1 (binds BRD4), a 1,8-diaminooctane linker and a thalidomide derivative (binds CRBN). Each block is a complete molecule; the definition file says which atom of each block forms the new bond and which hydrogen leaves (`ABSOLUTE` = atom indices in the `.sdf`).

In [2]:
print(open("input/dBET23.txt").read())
show_grid(["input/jq1.sdf", "input/c8diamine.sdf", "input/thal.sdf"],
          ["jq1 (BRD4)", "c8diamine (linker)", "thal (CRBN)"])

##[ BUILDING BLOCK ]
name	type	head_smart	head_bondor	head_deletor	end_smart	end_bondor	end_deletor
jq1	1	ABSOLUTE	6	32
c8diamine	2	ABSOLUTE	0	10	ABSOLUTE	9	28
thal	1	ABSOLUTE	1	23

##[ CONNECTION ]
name	mol_A	mol_B	repeat	reaction_times
part1	jq1	c8diamine	1	1
part2	part1	thal	1	1



[3D view - run the notebook in Jupyter to display it]


## 2. Assemble dBET23
`part1` = JQ1 + linker, `part2` = part1 + thalidomide.

In [3]:
TXT = "input/dBET23.txt"
for part in ("part1", "part2"):
    run(["PolyLego-BuildPolymer", "-i", "input", "-o", "output", "-t", TXT, "-l", part],
        log=f"output/build_{part}.log")

$ PolyLego-BuildPolymer -i input -o output -t input/dBET23.txt -l part1


  PolyLego 1.4.0 | BuildPolymer
  assembling part1 from input/dBET23.txt
  Finished in 0 s
    output/part1.sdf
    output/part1_graph.gml
  -> exit 0, 1 s
$ PolyLego-BuildPolymer -i input -o output -t input/dBET23.txt -l part2


  PolyLego 1.4.0 | BuildPolymer
  assembling part2 from input/dBET23.txt
  Finished in 0 s
    output/part2.sdf
    output/part2_graph.gml
  -> exit 0, 2 s


## 3. Assign force-field parameters

In [4]:
run(["PolyLego-AssignForceField", "-l", "input", "-t", TXT, "-o", "output",
     "-n", "part2", "-protac", "--gpu", GPU], log="output/assign.log")

$ PolyLego-AssignForceField -l input -t input/dBET23.txt -o output -n part2 -protac --gpu none


  PolyLego 1.4.0 | AssignForceField
  force field for part2 (PROTAC)
    building block jq1: GAFF-2.11 + AM1-BCC charges
    building block thal: GAFF-2.11 + AM1-BCC charges
    building block c8diamine: GAFF-2.11 + AM1-BCC charges
    junction parameters from capped fragments
    assembly topology
  Finished in 5 min 34 s
    output/assembly.top
    output/assembly.gro
    output/assembly.pdb
  -> exit 0, 336 s


In [5]:
import parmed
def topology_summary(top, gro):
    s = parmed.load_file(top, xyz=gro)
    n_imp = sum(d.improper for d in s.dihedrals) + len(s.impropers)
    print(f"atoms            {len(s.atoms)}")
    print(f"bonds            {len(s.bonds)}")
    print(f"angles           {len(s.angles)}")
    print(f"proper dihedrals {sum(not d.improper for d in s.dihedrals)}")
    print(f"impropers        {n_imp}")
    print(f"net charge       {sum(a.charge for a in s.atoms):+.4f} e")
    return s

_ = topology_summary("output/assembly.top", "output/assembly.gro")

  warnings.warn('%i 1-4 pairs were missing from the [ pairs ] '



atoms            107
bonds            113
angles           201
proper dihedrals 253
impropers        31
net charge       -0.0000 e


Is the assembled molecule the PROTAC in the crystal? Bond orders come from the assembly, the configuration of each stereocentre from the crystal coordinates. JQ1 and thalidomide each carry one stereocentre, and both must come out as in the building blocks.

In [6]:
from PolyLego.protac_dock.dockready import make_dockready_sdf
assembled = make_dockready_sdf("output", name="dBET23")
same, centres = compare_to_crystal_ligand(assembled, "input/protac_crystal.pdb")
print("same molecule as the crystal ligand:", same)
print(centres)
show_mol(assembled)

same molecule as the crystal ligand: True
{'assembled': [(15, 'S'), (36, 'S')], 'crystal': [(25, 'S'), (30, 'S')]}


[3D view - run the notebook in Jupyter to display it]


## 4. Binding sites
P2Rank pockets shipped in `input/pockets/` (P2Rank 2.4.1). How far are they from where dBET23 binds in the crystal?

In [7]:
import numpy as np
E3_SITE = pocket_center("input/pockets/e3.pdb_predictions.csv")
POI_SITE = pocket_center("input/pockets/poi.pdb_predictions.csv")
xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                for l in open("input/protac_crystal.pdb") if l.startswith("HETATM")])
for tag, site in (("CRBN", E3_SITE), ("BRD4", POI_SITE)):
    d = np.linalg.norm(xyz - np.array(site), axis=1).min()
    print(f"{tag:5s} pocket {np.round(site, 1)}  {d:4.1f} A to the nearest crystal PROTAC atom")

CRBN  pocket [64.4 46.9 47.2]   1.4 A to the nearest crystal PROTAC atom
BRD4  pocket [72.4 34.9 59. ]   0.9 A to the nearest crystal PROTAC atom


## 5. Ternary-complex poses + MD-ready inputs
The input is one fixed conformer of dBET23, `input/dBET23_conformer.sdf` (section 6 explains why it is fixed). No crystal coordinates of the complex enter the prediction.

In [8]:
run(["PolyLego-OptimizeBinding", "--protacs", "--md-ready",
     "--input_path", "input/", "--output_path", "output/",
     "--protein1", "e3.pdb", "--protein2", "poi.pdb",
     "--protac_mol", "input/dBET23_conformer.sdf",
     "--docking_site1", *E3_SITE, "--docking_site2", *POI_SITE,
     "--name", "dBET23", "--topn", 5], log="output/ternary.log", tail=12)

$ PolyLego-OptimizeBinding --protacs --md-ready --input_path input/ --output_path output/ --protein1 e3.pdb --protein2 poi.pdb --protac_mol input/dBET23_conformer.sdf --docking_site1 64.352 46.9484 47.2203 --docking_site2 72.401 34.9081 59.0102 --name dBET23 --topn 5


  PolyLego 1.4.0 | OptimizeBinding
  PROTAC ternary complex -> MD-ready inputs (top 5 poses)
  5 poses (pose source: polylego)
    rank  min nb (A)  pairs<2A   EM Fmax      QC
       1        1.32         6   skipped    fail
       2        1.69         1   skipped    fail
       3        2.02         0   skipped    fail
       4        1.32         3   skipped    fail
       5        2.02         0   skipped    fail
  QC reasons are in each pose's qc.json; next_steps.sh continues to MD.
  Finished in 2 min 23 s
    output/md_inputs
  -> exit 0, 144 s


Score every pose against the crystal:

In [9]:
import glob, pandas as pd
reference = write_reference_complex("input/poi.pdb", "input/e3.pdb", "output/reference.pdb")
man = json.load(open("output/md_inputs/manifest.json"))
rows = []
for p in man["poses"]:
    r = dockq(f"output/md_inputs/pose_{p['rank']:03d}/complex.pdb", reference, {"C"}, {"B"})
    q = p["qc"]
    rows.append({"rank": p["rank"], "DockQ": round(r["DockQ"], 3) if r else 0.0,
                 "fnat": round(r["fnat"], 3) if r else 0.0,
                 "iRMSD (A)": round(r["iRMSD"], 2) if r else None,
                 "closest contact (A)": round(q["min_nb_dist_A"], 2),
                 "warheads in pocket": q["warheads_in_pocket"]})
df = pd.DataFrame(rows)
df.insert(2, "CAPRI", [capri_label(v) for v in df["DockQ"]])
df

,rank,DockQ,CAPRI,fnat,iRMSD (A),closest contact (A),warheads in pocket
0,1,0.696,medium,0.833,1.68,1.32,True
1,2,0.042,incorrect,0.000,7.36,1.69,False
2,3,0.037,incorrect,0.000,12.90,2.02,False
3,4,0.046,incorrect,0.028,11.35,1.32,False
4,5,0.061,incorrect,0.056,11.01,2.02,False


The top-ranked pose (magenta BRD4) on the crystal arrangement, with CRBN as the common frame.

In [10]:
show_complex("output/reference.pdb", ligands=["output/md_inputs/pose_001/complex.pdb"],
             focus_ligand=False)

[3D view - run the notebook in Jupyter to display it]


**Quality control.** Each pose directory also holds `qc.json`, and its `pass` flag
is false for every pose here - including the top-ranked one, which is close to the crystal.
For that pose the two reasons are: six atom pairs between the molecules are closer than
2.0 A (the closest 1.3 A), which rigid-body docking leaves behind, and the optional
PROTAC-only minimisation was not run because no topology was passed. Minimise the solvated
complex before any MD; `next_steps.sh` in each pose directory lists the commands.

## 6. How much this result depends on the input

The top-ranked pose above is close to the crystal. That is a property of **this conformer**,
not something PolyLego achieves for dBET23 in general. Measured with this release on the
same inputs (P2Rank pockets, top 5 poses):

| input conformer | top-ranked pose CAPRI-acceptable (DockQ >= 0.23) |
|---|---|
| `dBET23_conformer.sdf` (this notebook) | yes - medium (0.70) |
| 9 other RDKit conformers of dBET23 (seeds 1-8 and the unrounded original) | 1 of 9 (DockQ 0.23) |
| `dBET23_conformer.sdf` with every atom moved by ~0.0001 A (8 random draws) | 3 of 8 |

Pooling the 45 poses from the 9 conformers and ranking them with the same score puts the
best one (DockQ 0.46) at position 26. With the pocket centres taken from the crystal ligand
instead of P2Rank, this conformer gives DockQ 0.80 (high).

What stays true: identical inputs give identical poses (checked across repeated runs and
thread counts). What does not: an arbitrary conformer of your PROTAC will not reproduce a
crystal arrangement - on 68 PROTAC ternary crystal structures the top-ranked pose was
CAPRI-acceptable in 2 of the 31 that could be docked from SMILES.

Treat PolyLego's ternary poses as candidates to be checked against other evidence.
`robustness.py` in this directory reruns the conformer comparison (about 15 minutes on
18 cores).